# Build YOLO26 TensorRT Engine For SqueakView

This notebook imports a YOLO26 pose `.pt` model into the `SqueakView` model package layout. It replaces the old `DeepStream-Yolo/artifacts` workflow with a self-contained package under `models/<model_name>/`.

Run top-to-bottom after editing the first config cell. The generated DeepStream config is YOLO26-pose-only and uses `NvDsInferParseYolo26Pose` from `native/nvdsinfer_custom_impl_yolo`.

The engine build path is intentionally local TensorRT only: Ultralytics exports ONNX on CPU, then `/usr/src/tensorrt/bin/trtexec` builds the `.engine`. Do not use Ultralytics `format="engine"` on this Jetson unless the PyTorch wheel is proven to support Orin `sm_87` CUDA kernels.

The generated DeepStream config writes paths relative to the config file so model packages remain portable across clone locations. The SqueakView GUI resolves those paths into a run-local config before launching DeepStream.

## 1. Configure The Model Import

Edit this cell for each model. The safest starting point is `.pt + data.yaml` from the same training run.

In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

# If the notebook is launched from build_engine/, use the parent repo root.
WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name in {"build_engine", "build-engine"}:
    WORKSPACE = WORKSPACE.parent

# ---- User-editable model inputs ---------------------------------------
# Source model. For a new import, point this to the YOLO26 .pt you copied over.
INPUT_MODEL = WORKSPACE / "build_me" / "stock_yolo26_pose" / "yolo26n-pose.pt"

# Training/data YAML. If None, the notebook tries dataset.yaml beside INPUT_MODEL,
# then the data path saved in the checkpoint train_args.
DATA_YAML = WORKSPACE / "build_me" / "stock_yolo26_pose" / "coco-pose.yaml"

# Output package name. Keep names filesystem-safe and explicit.
MODEL_NAME = "stock_yolo26n_pose_fp16"

# YOLO26 pose parser contract. This notebook intentionally supports pose only.
TASK = "pose"
PARSER_FUNC = "NvDsInferParseYolo26Pose"

# TensorRT/export settings. Batch size should match the intended DeepStream batch.
PRECISION = "fp16"  # fp32 or fp16; int8 needs a calibration workflow not implemented here.
BATCH_SIZE = 1
INFER_WIDTH = 640
INFER_HEIGHT = 640

# Local TensorRT settings. Engine builds always use trtexec, not Ultralytics format="engine".
TRT_DEVICE = 0
TRT_WORKSPACE_GIB = None  # None lets TensorRT choose. Use a float like 2.0 to cap workspace.
DYNAMIC_ENGINE = False

# DeepStream inference thresholds.
CONF_THRESHOLD = 0.25
NMS_IOU_THRESHOLD = 0.15
TOPK = 20
POSE_DRAW_THRESHOLD = 0.50

# Controls path style in model.yaml. The DeepStream nvinfer config writes
# paths relative to its config file so the package can move with the repo.
WRITE_RELATIVE_CONFIG_PATHS = True

# Keep existing package files unless this is explicitly enabled.
OVERWRITE_EXISTING = True

# Only needed if the ONNX has custom TensorRT plugin layers. The DeepStream
# parser library is not a TensorRT plugin and should not be loaded here.
INCLUDE_CUSTOM_PLUGIN_IN_TRTEXEC = False

# Notebook kernels can inherit Python/venv CUDA library paths that confuse the
# standalone TensorRT binary. Keep this enabled for Jetson builds.
SANITIZE_TRTEXEC_ENV = True

## 2. Resolve Package Paths

The generated package lives under `models/<MODEL_NAME>/` and does not write into the old `DeepStream-Yolo` tree.

In [2]:
INPUT_MODEL = Path(INPUT_MODEL).expanduser().resolve()
DATA_YAML = Path(DATA_YAML).expanduser().resolve() if DATA_YAML else None

if TASK != "pose":
    raise ValueError("SqueakView model builder currently supports YOLO26 pose models only.")
if PRECISION.lower() not in {"fp32", "fp16"}:
    raise ValueError("PRECISION must be 'fp32' or 'fp16'. INT8 needs a calibration workflow first.")
if BATCH_SIZE < 1:
    raise ValueError("BATCH_SIZE must be >= 1.")
if not INPUT_MODEL.exists():
    raise FileNotFoundError(f"INPUT_MODEL does not exist: {INPUT_MODEL}")

MODEL_ROOT = WORKSPACE / "models"
MODEL_DIR = MODEL_ROOT / MODEL_NAME
WEIGHTS_DIR = MODEL_DIR / "weights"
ONNX_DIR = MODEL_DIR / "onnx"
ENGINE_DIR = MODEL_DIR / "engines"
LABELS_DIR = MODEL_DIR / "labels"
CONFIG_DIR = MODEL_DIR / "configs"
VALIDATION_DIR = MODEL_DIR / "validation"

CUSTOM_LIB = WORKSPACE / "native" / "nvdsinfer_custom_impl_yolo" / "libnvdsinfer_custom_impl_Yolo.so"

for directory in (WEIGHTS_DIR, ONNX_DIR, ENGINE_DIR, LABELS_DIR, CONFIG_DIR, VALIDATION_DIR):
    directory.mkdir(parents=True, exist_ok=True)

source_stem = INPUT_MODEL.stem
artifact_stem = f"{source_stem}_{PRECISION.lower()}_b{BATCH_SIZE}"
PACKAGED_PT = WEIGHTS_DIR / INPUT_MODEL.name if INPUT_MODEL.suffix.lower() == ".pt" else None
ONNX_PATH = ONNX_DIR / f"{artifact_stem}.onnx"
ENGINE_PATH = ENGINE_DIR / f"{artifact_stem}.engine"
CONFIG_PATH = CONFIG_DIR / f"{MODEL_NAME}.txt"
POSE_SIDECAR_PATH = CONFIG_DIR / f"{MODEL_NAME}.pose.json"
MODEL_MANIFEST_PATH = MODEL_DIR / "model.yaml"
IMPORT_REPORT_PATH = VALIDATION_DIR / "import_report.json"

print("Workspace      :", WORKSPACE)
print("Model package  :", MODEL_DIR)
print("Input model    :", INPUT_MODEL)
print("Data YAML      :", DATA_YAML or "auto")
print("Parser library :", CUSTOM_LIB)
print("Config out     :", CONFIG_PATH)

Workspace      : /home/jetson/Documents/SqueakView
Model package  : /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16
Input model    : /home/jetson/Documents/SqueakView/build_me/stock_yolo26_pose/yolo26n-pose.pt
Data YAML      : /home/jetson/Documents/SqueakView/build_me/stock_yolo26_pose/coco-pose.yaml
Parser library : /home/jetson/Documents/SqueakView/native/nvdsinfer_custom_impl_yolo/libnvdsinfer_custom_impl_Yolo.so
Config out     : /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/configs/stock_yolo26n_pose_fp16.txt


## 3. Verify Build Tools

In [3]:
def find_trtexec() -> Path:
    candidates = [
        Path(os.environ.get("TRTEXEC") or ""),
        Path("/usr/src/tensorrt/bin/trtexec"),
        Path(shutil.which("trtexec") or ""),
    ]
    for path in candidates:
        if path and path.is_file() and os.access(path, os.X_OK):
            return path.resolve()
    raise FileNotFoundError("trtexec not found. Set TRTEXEC=/path/to/trtexec or install TensorRT tools.")

try:
    from ultralytics import YOLO
except ImportError as exc:
    raise RuntimeError("Ultralytics is required for .pt -> ONNX export. Open this notebook in the SqueakView environment.") from exc

TRTEXEC = find_trtexec()

if not CUSTOM_LIB.exists():
    raise FileNotFoundError(f"Custom YOLO parser library missing: {CUSTOM_LIB}")

print("engine builder:", "trtexec")
print("trtexec       :", TRTEXEC)
print("Ultralytics   :", YOLO)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/jetson/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
engine builder: trtexec
trtexec       : /usr/src/tensorrt/bin/trtexec
Ultralytics   : <class 'ultralytics.models.yolo.model.YOLO'>


/home/jetson/Documents/SqueakView/.venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## 4. CUDA Access Preflight

TensorRT engine builds require this notebook kernel to see the Jetson GPU. This catches missing group/session permissions, hidden CUDA devices, and CPU-only PyTorch before the engine build cell.

In [4]:
import grp


def current_group_names() -> list[str]:
    names = []
    for gid in os.getgroups():
        try:
            names.append(grp.getgrgid(gid).gr_name)
        except KeyError:
            names.append(str(gid))
    return names


def describe_device(path: str) -> dict[str, object]:
    p = Path(path)
    exists = p.exists()
    return {
        "path": path,
        "exists": exists,
        "readable": os.access(path, os.R_OK) if exists else False,
        "writable": os.access(path, os.W_OK) if exists else False,
    }

print("uid/gids:", os.getuid(), os.getgid(), current_group_names())
cuda_visible_devices = os.environ.get("CUDA_VISIBLE_DEVICES")
print("CUDA_VISIBLE_DEVICES:", cuda_visible_devices if cuda_visible_devices is not None else "<unset>")
if cuda_visible_devices == "":
    raise RuntimeError(
        "CUDA_VISIBLE_DEVICES is set to an empty string, which hides all CUDA devices. "
        "Unset it or set CUDA_VISIBLE_DEVICES=0 before launching Jupyter."
    )

jetson_cuda_devices = [
    "/dev/nvhost-gpu",
    "/dev/nvhost-ctrl-gpu",
    "/dev/nvmap",
    "/dev/nvidia0",
    "/dev/nvidiactl",
]
device_status = [describe_device(path) for path in jetson_cuda_devices]
for status in device_status:
    print(status)

required_nodes = ["/dev/nvhost-gpu", "/dev/nvmap"]
missing_or_blocked = [
    path for path in required_nodes
    if not Path(path).exists() or not os.access(path, os.R_OK | os.W_OK)
]
if missing_or_blocked:
    raise RuntimeError(
        "This notebook kernel cannot access Jetson CUDA device nodes: "
        + ", ".join(missing_or_blocked)
        + ". Make sure this user/session is in the video/render groups, then log out/in or restart Jupyter."
    )

try:
    import torch
    print("torch:", torch.__version__)
    print("torch.cuda.is_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        supported_arches = list(torch.cuda.get_arch_list()) if hasattr(torch.cuda, "get_arch_list") else []
        print("torch cuda device 0:", torch.cuda.get_device_name(0))
        print("torch cuda capability:", f"sm_{major}{minor}")
        print("torch compiled arches:", supported_arches or "<unknown>")
        print("torch CUDA note: not used for engine build; ONNX export is forced to CPU.")
except ImportError:
    print("torch: not installed; Ultralytics export will fail unless torch is installed.")

uid/gids: 1000 1000 ['adm', 'cdrom', 'sudo', 'audio', 'dip', 'video', 'plugdev', 'render', 'i2c', 'lpadmin', 'gdm', 'sambashare', 'weston-launch', 'gpio', 'jetson', 'flirimaging']
CUDA_VISIBLE_DEVICES: <unset>
{'path': '/dev/nvhost-gpu', 'exists': True, 'readable': True, 'writable': True}
{'path': '/dev/nvhost-ctrl-gpu', 'exists': True, 'readable': True, 'writable': True}
{'path': '/dev/nvmap', 'exists': True, 'readable': True, 'writable': True}
{'path': '/dev/nvidia0', 'exists': True, 'readable': True, 'writable': True}
{'path': '/dev/nvidiactl', 'exists': True, 'readable': True, 'writable': True}
torch: 2.12.1+cu126
torch.cuda.is_available: True
torch cuda device 0: Orin
torch cuda capability: sm_87
torch compiled arches: ['sm_80', 'sm_90']
torch CUDA note: not used for engine build; ONNX export is forced to CPU.


/home/jetson/Documents/SqueakView/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU0 Orin which is of compute capability (CC) 8.7.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
  _warn_unsupported_code(d, device_cc, code_ccs)


## 5. Read Model And YAML Metadata

This enforces the parts of the YOLO26 pose contract that can be validated before DeepStream runs.

In [5]:
import yaml

report: dict[str, object] = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "workspace": str(WORKSPACE),
    "model_name": MODEL_NAME,
    "source_model": str(INPUT_MODEL),
    "checks": [],
}


def add_check(name: str, ok: bool, detail: str = "") -> None:
    report["checks"].append({"name": name, "ok": bool(ok), "detail": detail})
    status = "PASS" if ok else "FAIL"
    print(f"[{status}] {name}: {detail}")
    if not ok:
        raise ValueError(f"Validation failed: {name}: {detail}")


def normalize_names(names_obj) -> list[str]:
    if names_obj is None:
        return []
    if isinstance(names_obj, dict):
        return [str(names_obj[k]) for k in sorted(names_obj, key=lambda x: int(x) if str(x).isdigit() else str(x))]
    if isinstance(names_obj, (list, tuple)):
        return [str(x) for x in names_obj]
    raise TypeError(f"Unsupported names format: {type(names_obj)!r}")


def load_yaml(path: Path | None) -> dict:
    if not path:
        return {}
    if not path.exists():
        raise FileNotFoundError(f"YAML file does not exist: {path}")
    data = yaml.safe_load(path.read_text()) or {}
    if not isinstance(data, dict):
        raise ValueError(f"Expected mapping in YAML: {path}")
    return data


def find_data_yaml_from_checkpoint(pt_path: Path, ckpt: dict) -> Path | None:
    candidates = [pt_path.parent / "dataset.yaml", pt_path.parent / "data.yaml"]
    train_args = ckpt.get("train_args") if isinstance(ckpt, dict) else None
    if isinstance(train_args, dict) and train_args.get("data"):
        candidates.append(Path(train_args["data"]).expanduser())
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if resolved.exists():
            return resolved
    return None

pt_metadata = {}
ckpt = None
if INPUT_MODEL.suffix.lower() == ".pt":
    import torch
    ckpt = torch.load(INPUT_MODEL, map_location="cpu", weights_only=False)
    model_obj = ckpt.get("model") if isinstance(ckpt, dict) else None
    if model_obj is None:
        raise ValueError("Checkpoint does not contain ckpt['model']; cannot infer YOLO metadata.")
    pt_metadata["names"] = normalize_names(getattr(model_obj, "names", None))
    pt_metadata["kpt_shape"] = list(getattr(model_obj, "kpt_shape", []) or [])
    if DATA_YAML is None:
        DATA_YAML = find_data_yaml_from_checkpoint(INPUT_MODEL, ckpt)

YAML_DATA = load_yaml(DATA_YAML)
yaml_names = normalize_names(YAML_DATA.get("names")) if YAML_DATA else []
class_names = yaml_names or pt_metadata.get("names", [])

kpt_shape = YAML_DATA.get("kpt_shape") or pt_metadata.get("kpt_shape") or []
kpt_shape = list(kpt_shape) if isinstance(kpt_shape, (list, tuple)) else []

def normalize_keypoint_names(names_obj, class_names: list[str]) -> list[str] | None:
    if names_obj is None:
        return None
    if isinstance(names_obj, (list, tuple)):
        return [str(x) for x in names_obj]
    if isinstance(names_obj, dict):
        # Ultralytics COCO pose stores names per class: kpt_names: {0: [nose, ...]}.
        if len(names_obj) == 1:
            only_value = next(iter(names_obj.values()))
            if isinstance(only_value, (list, tuple)):
                return [str(x) for x in only_value]
        for key in [0, "0", class_names[0] if class_names else None]:
            if key in names_obj and isinstance(names_obj[key], (list, tuple)):
                return [str(x) for x in names_obj[key]]
        flat = []
        for key in sorted(names_obj, key=lambda x: int(x) if str(x).isdigit() else str(x)):
            value = names_obj[key]
            if isinstance(value, (list, tuple)):
                flat.extend(str(x) for x in value)
            else:
                flat.append(str(value))
        return flat
    raise TypeError(f"Unsupported keypoint names format: {type(names_obj)!r}")

kp_names = normalize_keypoint_names(
    YAML_DATA.get("kp_names")
    or YAML_DATA.get("keypoint_names")
    or YAML_DATA.get("kpt_names")
    or YAML_DATA.get("keypoints"),
    class_names,
)

add_check("class_names_present", len(class_names) > 0, f"{len(class_names)} classes")
if YAML_DATA.get("nc") is not None:
    add_check("yaml_nc_matches_names", int(YAML_DATA["nc"]) == len(class_names), f"nc={YAML_DATA['nc']} names={len(class_names)}")
add_check("kpt_shape_present", len(kpt_shape) >= 2, f"kpt_shape={kpt_shape}")
keypoint_count = int(kpt_shape[0])
keypoint_dims = int(kpt_shape[1])
add_check("keypoint_dims_are_xy_conf", keypoint_dims == 3, f"kpt_shape={kpt_shape}")

if kp_names is None:
    kp_names = [f"kp_{idx:02d}" for idx in range(keypoint_count)]
    print("No named keypoints found in YAML; using generic kp_XX labels.")
add_check("keypoint_label_count_matches_shape", len(kp_names) == keypoint_count, f"labels={len(kp_names)} kpt_shape[0]={keypoint_count}")

print("Classes:", class_names)
print("Keypoints:", kp_names)
print("Data YAML:", DATA_YAML or "not found")

[PASS] class_names_present: 1 classes
[PASS] kpt_shape_present: kpt_shape=[17, 3]
[PASS] keypoint_dims_are_xy_conf: kpt_shape=[17, 3]
[PASS] keypoint_label_count_matches_shape: labels=17 kpt_shape[0]=17
Classes: ['person']
Keypoints: ['nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear', 'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle']
Data YAML: /home/jetson/Documents/SqueakView/build_me/stock_yolo26_pose/coco-pose.yaml


## 6. Copy Source Model And Write Labels

In [6]:
def copy_file_once(src: Path, dst: Path, overwrite: bool = False) -> None:
    if dst.exists() and not overwrite:
        print(f"Keeping existing file: {dst}")
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")

if INPUT_MODEL.suffix.lower() == ".pt":
    copy_file_once(INPUT_MODEL, PACKAGED_PT, overwrite=OVERWRITE_EXISTING)
    SOURCE_FOR_EXPORT = PACKAGED_PT
elif INPUT_MODEL.suffix.lower() == ".onnx":
    copy_file_once(INPUT_MODEL, ONNX_PATH, overwrite=OVERWRITE_EXISTING)
    SOURCE_FOR_EXPORT = None
else:
    raise ValueError("INPUT_MODEL must be .pt or .onnx")

classes_path = LABELS_DIR / "classes.txt"
labels_path = LABELS_DIR / "labels.txt"
if classes_path.exists() and not OVERWRITE_EXISTING:
    print(f"Keeping existing file: {classes_path}")
else:
    classes_path.write_text("\n".join(class_names) + "\n")
if labels_path.exists() and not OVERWRITE_EXISTING:
    print(f"Keeping existing file: {labels_path}")
else:
    labels_path.write_text("\n".join(kp_names) + "\n")

print("classes.txt:", classes_path)
print("labels.txt :", labels_path)

Copied /home/jetson/Documents/SqueakView/build_me/stock_yolo26_pose/yolo26n-pose.pt -> /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/weights/yolo26n-pose.pt
classes.txt: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/labels/classes.txt
labels.txt : /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/labels/labels.txt


## 7. Export ONNX

The ONNX artifact remains useful for DeepStream config fallback, shape inspection, and debugging even when the TensorRT engine is built through Ultralytics.

In [7]:
def move_exported_file(export_result, expected_suffix: str, destination: Path) -> Path:
    if isinstance(export_result, (list, tuple)):
        candidates = [Path(x) for x in export_result]
    else:
        candidates = [Path(str(export_result))]
    candidates.extend(SOURCE_FOR_EXPORT.parent.glob(f"{SOURCE_FOR_EXPORT.stem}*{expected_suffix}"))
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if candidate.exists() and candidate.suffix == expected_suffix:
            destination.parent.mkdir(parents=True, exist_ok=True)
            if candidate != destination.resolve():
                if destination.exists():
                    destination.unlink()
                shutil.move(str(candidate), str(destination))
            return destination
    raise FileNotFoundError(f"Could not find exported {expected_suffix} from {export_result!r}")

if INPUT_MODEL.suffix.lower() == ".pt":
    if ONNX_PATH.exists() and not OVERWRITE_EXISTING:
        print(f"Keeping existing ONNX: {ONNX_PATH}")
    else:
        model = YOLO(str(SOURCE_FOR_EXPORT))
        print("Exporting ONNX with Ultralytics...")
        export_result = model.export(
            format="onnx",
            imgsz=(INFER_HEIGHT, INFER_WIDTH),
            batch=BATCH_SIZE,
            simplify=True,
            nms=False,
            dynamic=DYNAMIC_ENGINE,
            device="cpu",
        )
        move_exported_file(export_result, ".onnx", ONNX_PATH)
        print("ONNX written:", ONNX_PATH)
else:
    print("ONNX input copied earlier:", ONNX_PATH)

add_check("onnx_exists", ONNX_PATH.exists(), str(ONNX_PATH))

Exporting ONNX with Ultralytics...
Ultralytics 8.4.84 🚀 Python-3.10.12 torch-2.12.1+cu126 CPU (ARMv8 Processor rev 1 (v8l))
YOLO26n-pose summary (fused): 132 layers, 2,926,494 parameters, 0 gradients, 7.5 GFLOPs

PyTorch: starting from '/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/weights/yolo26n-pose.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 57) (7.5 MB)

ONNX: starting export with onnx 1.22.0 opset 20...


/home/jetson/Documents/SqueakView/.venv/lib/python3.10/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...


2026-07-01 16:15:02.843141248 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"


ONNX: export success ✅ 8.0s, saved as '/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/weights/yolo26n-pose.onnx' (11.6 MB)

Export complete (10.3s)
Results saved to /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/weights/yolo26n-pose.onnx
Predict:         yolo predict task=pose model=/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/weights/yolo26n-pose.onnx imgsz=640 
Validate:        yolo val task=pose model=/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/weights/yolo26n-pose.onnx imgsz=640 data=/home/mason/code/ultralytics/ultralytics/cfg/datasets/coco-pose.yaml  
Visualize:       https://netron.app
ONNX written: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/onnx/yolo26n-pose_fp16_b1.onnx
[PASS] onnx_exists: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/onnx/yolo26n-pose_fp16_b1.onnx


## 8. Inspect ONNX Contract

The current YOLO26 pose parser expects each prediction row/channel to contain:

`x1, y1, x2, y2, confidence, class_id, keypoint_x/y/conf...`

So the output stride should be `6 + 3 * keypoint_count`.

In [8]:
import onnx

onnx_model = onnx.load(str(ONNX_PATH))
input_tensor = onnx_model.graph.input[0]
INPUT_NAME = input_tensor.name


def dim_to_value(dim) -> int | str:
    if dim.HasField("dim_value") and dim.dim_value > 0:
        return int(dim.dim_value)
    if dim.HasField("dim_param") and dim.dim_param:
        return str(dim.dim_param)
    return "dynamic"

input_shape = [dim_to_value(dim) for dim in input_tensor.type.tensor_type.shape.dim]
output_shapes = []
for output in onnx_model.graph.output:
    output_shapes.append({
        "name": output.name,
        "shape": [dim_to_value(dim) for dim in output.type.tensor_type.shape.dim],
    })

expected_stride = 6 + 3 * keypoint_count
stride_seen = False
unknown_output_dims = False
for output in output_shapes:
    for dim in output["shape"]:
        if isinstance(dim, int) and dim == expected_stride:
            stride_seen = True
        if not isinstance(dim, int):
            unknown_output_dims = True

if stride_seen:
    add_check("onnx_output_stride_matches_yolo26_pose", True, f"expected stride {expected_stride}; outputs={output_shapes}")
elif unknown_output_dims:
    report["checks"].append({
        "name": "onnx_output_stride_matches_yolo26_pose",
        "ok": None,
        "detail": f"dynamic output dims; expected stride {expected_stride}; outputs={output_shapes}",
    })
    print(f"[WARN] onnx_output_stride_matches_yolo26_pose: dynamic dims; expected stride {expected_stride}")
else:
    add_check("onnx_output_stride_matches_yolo26_pose", False, f"expected stride {expected_stride}; outputs={output_shapes}")

print("Input tensor:", INPUT_NAME, input_shape)
print("Outputs:", output_shapes)

[PASS] onnx_output_stride_matches_yolo26_pose: expected stride 57; outputs=[{'name': 'output0', 'shape': [1, 300, 57]}]
Input tensor: images [1, 3, 640, 640]
Outputs: [{'name': 'output0', 'shape': [1, 300, 57]}]


## 9. Build TensorRT Engine

Build the TensorRT engine with local TensorRT `trtexec`. This avoids Ultralytics `format="engine"` and does not depend on PyTorch CUDA kernel compatibility.

In [9]:
def onnx_has_dynamic_input(shape: list[int | str]) -> bool:
    return any(not isinstance(value, int) for value in shape)


def make_trtexec_env() -> dict[str, str]:
    env = os.environ.copy()
    if SANITIZE_TRTEXEC_ENV:
        # Avoid Python wheel CUDA libraries shadowing Jetson system CUDA/TensorRT.
        keep_paths = []
        for candidate in [
            "/usr/local/cuda-12.6/lib64",
            "/usr/lib/aarch64-linux-gnu/tegra",
            "/usr/lib/aarch64-linux-gnu",
            "/opt/nvidia/deepstream/deepstream/lib",
        ]:
            p = Path(candidate)
            if p.exists():
                keep_paths.append(str(p))
        for existing in env.get("LD_LIBRARY_PATH", "").split(os.pathsep):
            if not existing:
                continue
            if ".venv" in existing or "site-packages" in existing:
                continue
            if existing not in keep_paths:
                keep_paths.append(existing)
        env["LD_LIBRARY_PATH"] = os.pathsep.join(keep_paths)
    if env.get("CUDA_VISIBLE_DEVICES") == "":
        env.pop("CUDA_VISIBLE_DEVICES", None)
    if str(TRT_DEVICE).isdigit():
        env.setdefault("CUDA_VISIBLE_DEVICES", str(TRT_DEVICE))
    print("trtexec CUDA_VISIBLE_DEVICES:", env.get("CUDA_VISIBLE_DEVICES", "<unset>"))
    print("trtexec LD_LIBRARY_PATH:", env.get("LD_LIBRARY_PATH", "<unset>"))
    return env


def build_engine_with_trtexec() -> None:
    profile_shape = f"{BATCH_SIZE}x3x{INFER_HEIGHT}x{INFER_WIDTH}"
    min_shape = f"1x3x{INFER_HEIGHT}x{INFER_WIDTH}"

    cmd = [
        str(TRTEXEC),
        f"--onnx={ONNX_PATH}",
        f"--saveEngine={ENGINE_PATH}",
        f"--device={TRT_DEVICE}" if str(TRT_DEVICE).isdigit() else "",
    ]
    cmd = [part for part in cmd if part]

    if onnx_has_dynamic_input(input_shape):
        cmd.extend([
            f"--minShapes={INPUT_NAME}:{min_shape}",
            f"--optShapes={INPUT_NAME}:{profile_shape}",
            f"--maxShapes={INPUT_NAME}:{profile_shape}",
        ])

    if PRECISION.lower() == "fp16":
        cmd.append("--fp16")

    if INCLUDE_CUSTOM_PLUGIN_IN_TRTEXEC and CUSTOM_LIB.exists():
        cmd.append(f"--staticPlugins={CUSTOM_LIB}")

    if TRT_WORKSPACE_GIB is not None:
        workspace_mib = int(float(TRT_WORKSPACE_GIB) * 1024)
        cmd.append(f"--memPoolSize=workspace:{workspace_mib}")

    if ENGINE_PATH.exists() and not OVERWRITE_EXISTING:
        print(f"Keeping existing engine: {ENGINE_PATH}")
        return

    print("Running:", " ".join(str(x) for x in cmd))
    result = subprocess.run(cmd, cwd=str(WORKSPACE), text=True, env=make_trtexec_env())
    print("Return code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError("trtexec failed; check above logs.")


build_engine_with_trtexec()
add_check("engine_exists", ENGINE_PATH.exists(), str(ENGINE_PATH))

Running: /usr/src/tensorrt/bin/trtexec --onnx=/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/onnx/yolo26n-pose_fp16_b1.onnx --saveEngine=/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/engines/yolo26n-pose_fp16_b1.engine --device=0 --fp16
trtexec CUDA_VISIBLE_DEVICES: 0
trtexec LD_LIBRARY_PATH: /usr/local/cuda-12.6/lib64:/usr/lib/aarch64-linux-gnu/tegra:/usr/lib/aarch64-linux-gnu:/opt/nvidia/deepstream/deepstream/lib
&&&& RUNNING TensorRT.trtexec [TensorRT v100300] # /usr/src/tensorrt/bin/trtexec --onnx=/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/onnx/yolo26n-pose_fp16_b1.onnx --saveEngine=/home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/engines/yolo26n-pose_fp16_b1.engine --device=0 --fp16
[07/01/2026-16:15:19] [I] === Model Options ===
[07/01/2026-16:15:19] [I] Format: ONNX
[07/01/2026-16:15:19] [I] Model: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/onnx/yolo26n-pose_fp16_b1.onnx
[07/01/2026-

[07/01/2026-16:24:31] [W] * GPU compute time is unstable, with coefficient of variance = 3.26813%.
[07/01/2026-16:24:31] [W]   If not already in use, locking GPU clock frequency or adding --useSpinWait may improve the stability.


Return code: 0
[PASS] engine_exists: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/engines/yolo26n-pose_fp16_b1.engine


## 10. Generate DeepStream Config, Pose Sidecar, And Manifest

The DeepStream config intentionally contains only keys that `nvinfer` understands. SqueakView reads pose metadata from the `.pose.json` sidecar.

In [10]:
def rel_or_abs(path: Path) -> str:
    path = path.resolve()
    if WRITE_RELATIVE_CONFIG_PATHS:
        try:
            return path.relative_to(WORKSPACE).as_posix()
        except ValueError:
            pass
    return str(path)


def path_for_deepstream_config(path: Path) -> str:
    # nvinfer resolves file keys relative to the config file path. Keep those
    # paths relative to CONFIG_PATH.parent so the package survives repo moves.
    rel = os.path.relpath(path.resolve(), CONFIG_PATH.parent.resolve())
    return rel.replace(os.sep, "/")

network_mode = 2 if PRECISION.lower() == "fp16" else 0

config_text = f"""[property]
gpu-id=0
net-scale-factor=0.0039215697906911373
model-color-format=0

onnx-file={path_for_deepstream_config(ONNX_PATH)}
model-engine-file={path_for_deepstream_config(ENGINE_PATH)}

network-mode={network_mode}
network-type=0
infer-dims=3;{INFER_WIDTH};{INFER_HEIGHT}
batch-size={BATCH_SIZE}
output-tensor-meta=1

num-detected-classes={len(class_names)}
labelfile-path={path_for_deepstream_config(classes_path)}
parse-bbox-func-name={PARSER_FUNC}
custom-lib-path={path_for_deepstream_config(CUSTOM_LIB)}
engine-create-func-name=NvDsInferYoloCudaEngineGet

cluster-mode=2
maintain-aspect-ratio=1
symmetric-padding=1
workspace-size=2048
gie-unique-id=1
interval=0
process-mode=1

[class-attrs-all]
nms-iou-threshold={NMS_IOU_THRESHOLD}
pre-cluster-threshold={CONF_THRESHOLD}
topk={TOPK}
"""

if CONFIG_PATH.exists() and not OVERWRITE_EXISTING:
    print(f"Keeping existing config: {CONFIG_PATH}")
else:
    CONFIG_PATH.write_text(config_text)
    print("Config written:", CONFIG_PATH)

pose_sidecar = {
    "schema_version": 1,
    "task": "pose",
    "parser": PARSER_FUNC,
    "keypoint_labels_path": "../labels/labels.txt",
    "keypoint_count": keypoint_count,
    "keypoint_dims": keypoint_dims,
    "draw_threshold": POSE_DRAW_THRESHOLD,
}
if POSE_SIDECAR_PATH.exists() and not OVERWRITE_EXISTING:
    print(f"Keeping existing pose sidecar: {POSE_SIDECAR_PATH}")
else:
    POSE_SIDECAR_PATH.write_text(json.dumps(pose_sidecar, indent=2) + "\n")
    print("Pose sidecar written:", POSE_SIDECAR_PATH)

manifest = {
    "schema_version": 1,
    "name": MODEL_NAME,
    "framework": "yolo26",
    "task": "pose",
    "precision": PRECISION.lower(),
    "batch_size": BATCH_SIZE,
    "engine_builder": "trtexec",
    "dynamic_engine": DYNAMIC_ENGINE,
    "input_shape": [3, INFER_HEIGHT, INFER_WIDTH],
    "classes": class_names,
    "keypoints": kp_names,
    "parser_contract": {
        "parse_bbox_func": PARSER_FUNC,
        "expected_output_stride": expected_stride,
        "custom_lib": rel_or_abs(CUSTOM_LIB),
        "engine_create_func": "NvDsInferYoloCudaEngineGet",
    },
    "files": {
        "config": rel_or_abs(CONFIG_PATH),
        "pose_sidecar": rel_or_abs(POSE_SIDECAR_PATH),
        "weights": rel_or_abs(PACKAGED_PT) if PACKAGED_PT else None,
        "onnx": rel_or_abs(ONNX_PATH),
        "engine": rel_or_abs(ENGINE_PATH),
        "classes": rel_or_abs(classes_path),
        "keypoint_labels": rel_or_abs(labels_path),
        "import_report": rel_or_abs(IMPORT_REPORT_PATH),
    },
    "source": {
        "model": str(INPUT_MODEL),
        "data_yaml": str(DATA_YAML) if DATA_YAML else None,
    },
    "thresholds": {
        "confidence": CONF_THRESHOLD,
        "nms_iou": NMS_IOU_THRESHOLD,
        "topk": TOPK,
        "pose_draw": POSE_DRAW_THRESHOLD,
    },
}

if MODEL_MANIFEST_PATH.exists() and not OVERWRITE_EXISTING:
    print(f"Keeping existing manifest: {MODEL_MANIFEST_PATH}")
else:
    MODEL_MANIFEST_PATH.write_text(yaml.safe_dump(manifest, sort_keys=False))
    print("Manifest written:", MODEL_MANIFEST_PATH)

report.update({
    "model_dir": str(MODEL_DIR),
    "config": str(CONFIG_PATH),
    "pose_sidecar": str(POSE_SIDECAR_PATH),
    "manifest": str(MODEL_MANIFEST_PATH),
    "onnx": str(ONNX_PATH),
    "engine": str(ENGINE_PATH),
    "classes": class_names,
    "keypoints": kp_names,
    "input_shape": [BATCH_SIZE, 3, INFER_HEIGHT, INFER_WIDTH],
    "engine_builder": "trtexec",
    "dynamic_engine": DYNAMIC_ENGINE,
    "expected_output_stride": expected_stride,
    "onnx_input": {"name": INPUT_NAME, "shape": input_shape},
    "onnx_outputs": output_shapes,
})
IMPORT_REPORT_PATH.write_text(json.dumps(report, indent=2) + "\n")
print("Import report written:", IMPORT_REPORT_PATH)

Config written: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/configs/stock_yolo26n_pose_fp16.txt
Pose sidecar written: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/configs/stock_yolo26n_pose_fp16.pose.json
Manifest written: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/model.yaml
Import report written: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/validation/import_report.json


## 11. Final Checks

These checks are fast and catch path mistakes before the GUI tries to start DeepStream.

In [11]:
required_files = [
    CONFIG_PATH,
    POSE_SIDECAR_PATH,
    MODEL_MANIFEST_PATH,
    ONNX_PATH,
    ENGINE_PATH,
    classes_path,
    labels_path,
    IMPORT_REPORT_PATH,
]
if PACKAGED_PT:
    required_files.append(PACKAGED_PT)

for path in required_files:
    add_check(f"exists:{path.relative_to(MODEL_DIR) if path.is_relative_to(MODEL_DIR) else path.name}", path.exists(), str(path))

import configparser

cfg_text = CONFIG_PATH.read_text()
cfg = configparser.ConfigParser(interpolation=None)
cfg.optionxform = str
cfg.read_string(cfg_text)
property_cfg = cfg["property"]

add_check("config_uses_yolo26_pose_parser", f"parse-bbox-func-name={PARSER_FUNC}" in cfg_text, PARSER_FUNC)
add_check("config_batch_matches_manifest", f"batch-size={BATCH_SIZE}" in cfg_text, f"batch-size={BATCH_SIZE}")
add_check("config_has_no_legacy_pose_keys", "pose-kpt-labels-path" not in cfg_text and "pose-draw-threshold" not in cfg_text, "pose metadata lives in .pose.json")

for key in ["onnx-file", "model-engine-file", "custom-lib-path", "labelfile-path"]:
    configured_raw = property_cfg[key]
    configured = Path(configured_raw)
    resolved = configured if configured.is_absolute() else (CONFIG_PATH.parent / configured).resolve()
    add_check(f"config_{key}_is_relative", not configured.is_absolute(), str(configured))
    add_check(f"config_{key}_resolves", resolved.exists(), f"{configured_raw} -> {resolved}")

pose_meta = json.loads(POSE_SIDECAR_PATH.read_text())
add_check("pose_sidecar_keypoint_count_matches_yaml", int(pose_meta["keypoint_count"]) == keypoint_count, f"pose={pose_meta['keypoint_count']} yaml={keypoint_count}")
add_check("pose_sidecar_keypoint_dims_matches_yaml", int(pose_meta["keypoint_dims"]) == keypoint_dims, f"pose={pose_meta['keypoint_dims']} yaml={keypoint_dims}")

IMPORT_REPORT_PATH.write_text(json.dumps(report, indent=2) + "\n")

print("\nModel package ready:", MODEL_DIR)
print("DeepStream config:", CONFIG_PATH)
print("To select manually in an experiment profile, use:")
print(CONFIG_PATH)
print("\nJetson note: after building engines in this notebook, close Jupyter or reboot before long DeepStream acquisition runs if tegrastats reports low NvMap lfb memory.")

[PASS] exists:configs/stock_yolo26n_pose_fp16.txt: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/configs/stock_yolo26n_pose_fp16.txt
[PASS] exists:configs/stock_yolo26n_pose_fp16.pose.json: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/configs/stock_yolo26n_pose_fp16.pose.json
[PASS] exists:model.yaml: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/model.yaml
[PASS] exists:onnx/yolo26n-pose_fp16_b1.onnx: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/onnx/yolo26n-pose_fp16_b1.onnx
[PASS] exists:engines/yolo26n-pose_fp16_b1.engine: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/engines/yolo26n-pose_fp16_b1.engine
[PASS] exists:labels/classes.txt: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/labels/classes.txt
[PASS] exists:labels/labels.txt: /home/jetson/Documents/SqueakView/models/stock_yolo26n_pose_fp16/labels/labels.txt
[PASS] exists:validation/import_report.json: /home/jet